In [ ]:
from datetime import datetime

import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import mlflow
import shap

from sklearn.metrics import mean_absolute_error

%matplotlib inline

1.データの読み込み

In [ ]:
df_raw = pd.read_csv("../data/processed/dataset/baseline.csv")

In [ ]:
df_raw.head(5)

2.特徴量エンジニアリング

In [ ]:
df_processed = df_raw.copy()

In [ ]:
# 特徴量シャッフルもやる？
# shap値とかみてるし、一旦保留。

2.1時間特徴量

In [ ]:
df_processed["datetime"] = pd.to_datetime(df_processed["datetime"])
df_processed["month"] = df_processed["datetime"].dt.month.astype(int)
df_processed["hour"] = df_processed["datetime"].dt.hour.astype(int)

In [ ]:
# dayofyear
df_processed["day_of_year"] = df_processed["datetime"].dt.day_of_year

In [ ]:
# dayofweek
# →曜日を入れる。
df_processed["day_of_week"] = df_processed["datetime"].dt.day_of_week


In [ ]:
# sincos化

# うるう年の処理を含めて、day of yearをsin/cosにする

is_leap = df_processed["datetime"].dt.is_leap_year

max_days = np.where(is_leap, 366, 365)

normalized_day_of_year = (df_processed["day_of_year"] - 1) / max_days

 # sin/cos変換
df_processed["day_of_year_sin"] = np.sin(2 * np.pi * normalized_day_of_year.reset_index(drop=True))
df_processed["day_of_year_cos"] = np.cos(2 * np.pi * normalized_day_of_year.reset_index(drop=True))

In [ ]:
# sincos化関数
def sin_cos(sr, duration, prefix):

    sin = np.sin(2 * np.pi * sr / duration)
    cos = np.cos(2 * np.pi * sr / duration)

    output = pd.DataFrame({
        f"{prefix}_sin": sin,
        f"{prefix}_cos": cos
    })

    return output

In [ ]:
# hourのsincos
df = pd.DataFrame()
df = sin_cos(df_processed["hour"], 24, "hour")
df_processed = pd.concat([df_processed, df], axis=1)

In [ ]:
# day_of_weekのsincos
df = pd.DataFrame()
df = sin_cos(df_processed["day_of_week"], 7, "day_of_week")
df_processed = pd.concat([df_processed, df], axis=1)

In [ ]:
df_processed.head(5)

2.2気温の特徴量

In [ ]:
# ラグ
feature_name = "temperature"
list_add_lag_names = []


for col_name in df_processed.filter(like=feature_name).columns:
    
    lag1 = f"{col_name}_lag1"
    lag2 = f"{col_name}_lag2"
    lag3 = f"{col_name}_lag3"

    # 1時間前の値との差分
    # 差分があった方が精度がいいから入れる
    # 気温差が大きい方が、電力需要の変化が大きい可能性
    delta_lag1 = f"{col_name}_delta_lag1"

    df_processed[lag1] = df_processed[col_name].shift(1)
    df_processed[lag2] = df_processed[col_name].shift(2)
    df_processed[lag3] = df_processed[col_name].shift(3)

    df_processed[delta_lag1] = df_processed[col_name] - df_processed[lag1]

    # list_add_lag_names.extend([lag1, lag2, lag3])
    list_add_lag_names.extend([lag1, lag2, lag3, delta_lag1])


In [ ]:
# 追加した特徴量の名前の確認
list_add_lag_names

In [ ]:
# 移動平均
feature_name = "temperature"
list_add_sma_names = []


for col_name in df_processed.filter(like=feature_name).columns:
    
    # ラグを弾く
    if "lag" not in col_name:
        sma3 = f"{col_name}_sma3"
        sma5 = f"{col_name}_sma5"

        # 1時間前の値との差分
        # 差分があったほうが精度がいいから入れる
        # 気温差が大きい方が、電力需要の変化につながる可能性
        delta_sma3 = f"{col_name}_delta_sma3"
        delta_sma5 = f"{col_name}_delta_sma5"

        df_processed[sma3] = df_processed[col_name].rolling(window=3).mean()
        df_processed[sma5] = df_processed[col_name].rolling(window=5).mean()

        df_processed[delta_sma3] = df_processed[col_name] - df_processed[sma3]
        df_processed[delta_sma5] = df_processed[col_name] - df_processed[sma5]


        # list_add_sma_names.extend([sma3, sma5])
        list_add_sma_names.extend([sma3, sma5, delta_sma3, delta_sma5])

2.3気温以外の気象特徴量

In [ ]:
# 気温以外に試す特徴量

list_weather_features = ["precipitation",  "sunshine_duration", "wind_speed", "wind_direction", "dew_point", 
 "vapor_pressure", "humidity", "local_pressure", "sea_level_pressure", 
 "weather", "visibility"]

dict_weather_features = {}

for feature in list_weather_features:
    
    list_cols = []
    for col in df_processed.filter(like=feature).columns:
        if "_flag" not in col:
            list_cols.append(col)

    dict_weather_features[feature] = list_cols



In [ ]:
# 質的変数はカラムタイプをカテゴリに変更

# 風向き
for col_name in dict_weather_features["wind_direction"]:
        df_processed[col_name] = df_processed[col_name].astype("category")

# 天候（晴れとか、曇りとか）
for col_name in dict_weather_features["weather"]:
        df_processed[col_name] = df_processed[col_name].astype("category")

2.4特徴量作成

In [ ]:
# 不快指数を計算
# discomfort index = 0.81 * 気温 + 0.01*湿度*(0.99*気温-14.3) + 46.3

list_pref_names = ["chiba", "gumma", "ibaraki", "kanagawa", "saitama", "tochigi", "tokyo", "yamanashi"]

list_col_name_discomfort_index = []

for pref in list_pref_names:
    df_processed[f"{pref}_discomfort_index"] = (0.81*df_processed[f"{pref}_temperature"] 
                                                + 0.01*df_processed[f"{pref}_humidity"]*(0.99*df_processed[f"{pref}_temperature"]-14.3) 
                                                + 46.3)
    list_col_name_discomfort_index.append(f"{pref}_discomfort_index")

3.mlflowを用いて学習

In [ ]:
def make_dataset(df_processed: pd.DataFrame, x_cols: list, y_col: str):

    # 学習データの期間: 2021/01/01 00:00 ~ 2023/12/31 23:00
    # 検証データの期間: 2024/01/01 00:00 ~ 2024/12/31 23:00
    # テストデータの期間: 2025/01/01 00:00 ~ 2025/12/31 23:00

    val_start =  datetime(2024, 1, 1, 0, 0, 0)
    test_start =  datetime(2025, 1, 1, 0, 0, 0)

    if not isinstance(df_processed["datetime"], datetime):
        df_processed["datetime"] = pd.to_datetime(df_processed["datetime"])

    df_train = df_processed[df_processed["datetime"] < val_start].copy()
    df_val = df_processed[(df_processed["datetime"] >= val_start) & (df_processed["datetime"] < test_start)].copy()
    df_test = df_processed[df_processed["datetime"] >= test_start].copy()

    # 学習、検証、テストに分割
    X_train = df_train[x_cols]
    y_train =df_train[y_col]
    y_train.name = y_col

    X_val = df_val[x_cols]
    y_val = df_val[y_col]
    y_val.name = y_col

    X_test = df_test[x_cols]
    y_test = df_test[y_col]
    y_test.name = y_col

    return X_train, y_train, X_val, y_val, X_test, y_test

In [ ]:
# 学習に使う特徴量

# ベース
x_cols = ["chiba_temperature", "gumma_temperature","ibaraki_temperature", "kanagawa_temperature", "saitama_temperature",
       "tochigi_temperature", "tokyo_temperature", "yamanashi_temperature",
       "hour_sin", "hour_cos", "day_of_week_sin", "day_of_week_cos", "day_of_year_sin", "day_of_year_cos"
       ]

# ここから下で、追加する特徴量名を追加

# ラグ
x_cols.extend(list_add_lag_names)

# 移動平均
x_cols.extend(list_add_sma_names)

# ↑ここまでは基本で入れる.天候特徴量を色々と入れても、feature_importance, shap値ともに、時間系と気温の影響は大きいため。

# ↓下記は一個一個試してみて、精度があがるのを残す

# 相関が強そうな特徴量を一部消してみる
# set1: dew_point,vapor_pressure, humidity
# set2: weather, visibility
# →外さずに、入れた方が精度良さそう

# 降水量
x_cols.extend(dict_weather_features["precipitation"])

# 日照時間
x_cols.extend(dict_weather_features["sunshine_duration"])

# # 風速→使わない
# x_cols.extend(dict_weather_features["wind_speed"])

# # 風向き→使わない
# x_cols.extend(dict_weather_features["wind_direction"])

# # 露点→set1
x_cols.extend(dict_weather_features["dew_point"])

# # 蒸気圧→set1
x_cols.extend(dict_weather_features["vapor_pressure"])

# # 湿度→set1
# x_cols.extend(dict_weather_features["humidity"])

# # 現地気圧→使わない
# x_cols.extend(dict_weather_features["local_pressure"])

# # 海面気圧→使わない
# x_cols.extend(dict_weather_features["sea_level_pressure"])

# # 天候→set2
x_cols.extend(dict_weather_features["weather"])

# # 視程→set2
x_cols.extend(dict_weather_features["visibility"])


# ここからしたは自作特徴量
x_cols.extend(list_col_name_discomfort_index)

In [ ]:
# feature importanceのgainが0を除外
# list_remove_feature_importance = ["tokyo_temperature_delta_sma3", "gumma_precipitation", "kanagawa_precipitation", 
#                                   "yamanashi_temperature_delta_lag1", "saitama_precipitation", "yamanashi_precipitation",
#                                   "gumma_temperature_delta_sma3", "chiba_temperature_delta_lag1"]

# x_cols = [i for i in x_cols if i not in list_remove_feature_importance]

In [ ]:
    
# trainの設定
y_col = "demand"
x_cols = x_cols # 特徴量が増えるとごちゃごちゃするから、特徴量カラムの設定は一個上のセルに分ける。
run_name = "exp028_Add:discomfort_index_Remove:humidity"
memo = "不快指数追加。humidity削除。"

# データセット作成
X_train, y_train, X_val, y_val, X_test, y_test = make_dataset(df_processed=df_processed, 
                                                                x_cols=x_cols, 
                                                                y_col=y_col)


# モデルのインスタンス作成
model = lgb.LGBMRegressor(
        random_state = 42,
    )

# データセット作成
eval_set = [(X_val.reset_index(drop=True), y_val.reset_index(drop=True))]


# MLflowでログ保存ここから

# mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment(
    "electricity-demand-forecast"
)

# アクティブなRunが残っていれば強制終了する
# エラー防止
if mlflow.active_run():
    mlflow.end_run()


with mlflow.start_run(run_name=run_name):

    mlflow.log_param("n_features", len(x_cols))

    mlflow.log_text(
        "\n".join(x_cols),
        "features.txt"
    )

    mlflow.set_tag("memo", memo)

    model.fit(
        X_train, 
        y_train,
        eval_set = eval_set,
        # verbose=-1 # 学習ログを省略
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=True)
            ]
    )

    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    print("MAE:", mae)
    mlflow.log_metric("mae", mae)

    # feature_importance
    importances = model.booster_.feature_importance(importance_type="gain")
    df_feature_importance = pd.DataFrame({"feature": x_cols, "importance": importances})
    df_feature_importance = df_feature_importance.sort_values(by="importance", ascending=False).reset_index(drop=True)

    path_feature_importance = "../data/tmp/feature_importance.csv"
    df_feature_importance.to_csv(path_feature_importance, index=False)
    mlflow.log_artifact(path_feature_importance)

   # SHAP値
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)

    if hasattr(shap_values, "values"):
        # 新しいSHAPオブジェクト（Explanationオブジェクト）の場合
        shap_matrix = shap_values.values
    else:
        # 従来のNumPy配列の場合
        shap_matrix = shap_values

    df_shap = pd.DataFrame(shap_matrix, columns=X_test.columns)

    path_shap_csv = "../data/tmp/shap.csv"
    df_shap.to_csv(path_shap_csv, index=False)
    mlflow.log_artifact(path_shap_csv)

    path_shap_png = "../data/tmp/shap.png"
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_matrix, X_test, show=False)
    plt.tight_layout()
    plt.savefig(path_shap_png)
    plt.close() # ノートブック上に図の表示をしない
    mlflow.log_artifact(path_shap_png)



In [ ]:
# print(mlflow.__version__)

4.mlflowで学習履歴を確認

In [ ]:
# CLI→ mlflow ui